# 02 索引、切片与布尔筛选

> ### 本章目标
>
> - 掌握一维数组的索引与切片（负索引、步长、反转）
> - 掌握多维数组的 `m[i, j]` 索引与行 / 列 / 子块切片
> - 深刻理解 **切片是视图（view）不是拷贝** 这一核心概念，并能用 `np.shares_memory` 验证
> - 掌握花式索引（整数数组索引）与布尔索引，以及它们与切片的区别
> - 掌握 `np.where` 的三种用法（返回索引 / 向量化 if-else / 嵌套多分支）
> - 掌握切片赋值与广播赋值的技巧
>
> 📌 建议：上一章的 ndarray 属性（shape、dtype）是本章基础，可先快速复习。

---

In [1]:
import numpy as np

print(np.__version__)

2.4.4


## 2.1 一维数组的索引与切片

一维数组的索引和切片语法与 Python 内置 `list` **完全一致**，可以放心沿用已有经验。

**要点回顾：**

| 写法 | 含义 |
|---|---|
| `a[i]` | 取第 i 个元素（从 0 开始） |
| `a[-i]` | 从末尾倒数第 i 个 |
| `a[s:e]` | 切片：含 s，不含 e（左闭右开） |
| `a[s:e:step]` | 带步长的切片 |
| `a[::-1]` | 反转数组 |

> 💡 **提示**：切片越界不会报错，NumPy 会自动截断到合法范围。

In [2]:
a = np.arange(10)
print('a =', a)

# 正索引
print('a[0]  =', a[0])
print('a[4]  =', a[4])

# 负索引（从末尾倒数）
print('a[-1] =', a[-1])   # 最后一个元素
print('a[-3] =', a[-3])   # 倒数第 3 个

# 切片 [start:stop:step]，start 含、stop 不含
print('a[2:6]  =', a[2:6])
print('a[:4]   =', a[:4])
print('a[4:]   =', a[4:])
print('a[::2]  =', a[::2])     # 步长 2
print('a[1::2] =', a[1::2])
print('a[::-1] =', a[::-1])    # 反转

# 切片越界不报错，自动截断
print('a[2:100] =', a[2:100])

a = [0 1 2 3 4 5 6 7 8 9]
a[0]  = 0
a[4]  = 4
a[-1] = 9
a[-3] = 7
a[2:6]  = [2 3 4 5]
a[:4]   = [0 1 2 3]
a[4:]   = [4 5 6 7 8 9]
a[::2]  = [0 2 4 6 8]
a[1::2] = [1 3 5 7 9]
a[::-1] = [9 8 7 6 5 4 3 2 1 0]
a[2:100] = [2 3 4 5 6 7 8 9]


## 2.2 多维数组的索引：`m[i, j]` vs `m[i][j]`

对于二维数组 `m`，取元素有**两种**等价写法：

```mermaid
flowchart LR
    A[m i j 一次定位] --> F[推荐 写法简洁更快]
    C[m i 先取第 i 行] --> D[生成中间行数组]
    D --> E[再取第 j 个元素]
    E --> G[多一步 性能略差]
```

- `m[i, j]`：一次完成，直接定位到元素；
- `m[i][j]`：先 `m[i]` 得到第 i 行（会生成中间对象），再取第 j 个。

> 💡 **提示**：写法上两者结果相同，但**推荐始终用逗号形式 `m[i, j]`**——更符合 NumPy 语义、更快、也更易读。

**切片提取行 / 列 / 子块：**

| 写法 | 含义 | 结果 shape |
|---|---|---|
| `m[1]` | 第 1 行 | `(4,)` 一维 |
| `m[:, 1]` | 第 1 列 | `(3,)` 一维 |
| `m[:, 1:2]` | 第 1 列（保持二维） | `(3, 1)` |
| `m[:2, 1:]` | 前两行 × 后三列的子块 | `(2, 3)` |

In [3]:
m = np.arange(12).reshape(3, 4)
print('m ：')
print(m)

# 取单个元素：两种等价写法
print('m[1, 2] =', m[1, 2])   # 推荐：一次定位
print('m[1][2] =', m[1][2])   # 不推荐：两步（先生成中间行数组）

# 取整行 / 整列
print('m[1]      =', m[1])        # 第 1 行
print('m[:, 1]   =', m[:, 1])     # 第 1 列（注意 shape 变成一维）
print('m[:, 1:2] ：')
print(m[:, 1:2])                  # 第 1 列（用切片写法保持二维）

# 取子块
print('m[:2, 1:] ：')
print(m[:2, 1:])                  # 前两行、后三列

m ：
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
m[1, 2] = 6
m[1][2] = 6
m[1]      = [4 5 6 7]
m[:, 1]   = [1 5 9]
m[:, 1:2] ：
[[1]
 [5]
 [9]]
m[:2, 1:] ：
[[1 2 3]
 [5 6 7]]


## 2.3 ⚠️ 核心概念：切片是「视图」不是「拷贝」

这是 NumPy 最容易踩的坑，**没有之一**。

**切片 `a[s:e]` 返回的是原数组的一个「视图」（view）——它不复制数据，而是和原数组共享同一块内存。** 因此修改切片，原数组会被一起改变！

```mermaid
flowchart LR
    subgraph 共享同一块内存
        A[a 创建数组] --> M[连续内存块 0 1 2 3 4 5]
        B[sub 切片 a 2到5] --> M
    end
    subgraph 修改效果
        C[修改 sub 的元素] --> D[内存里的值被改写]
        D --> E[a 也跟着变]
    end
```

**与 Python list 的关键对比：**

| 操作 | Python list | NumPy ndarray |
|---|---|---|
| `x[2:5]` 返回 | **拷贝**（独立数据） | **视图**（共享内存） |
| 修改切片 | 原列表不变 | **原数组跟着变** |
| 验证方式 | — | `np.shares_memory(a, sub)` |

> ⚠️ **陷阱**：如果你想把切片「另存一份自己玩」，必须显式调用 `.copy()`。忘记这一步，就会在不知不觉中改坏原数组——尤其在把切片传给函数时最危险。

In [4]:
# 视图：修改切片会改原数组
a = np.arange(6)
sub = a[2:5]
print('a   =', a)
print('sub =', sub)

sub[0] = 999   # 修改“切片”
print('修改 sub[0]=999 之后 a =', a)          # 原数组也被改了！
print('np.shares_memory(a, sub) =', np.shares_memory(a, sub))  # True

# 与 Python list 对比：list 切片返回【拷贝】
lst = [0, 1, 2, 3, 4, 5]
lst_sub = lst[2:5]
lst_sub[0] = 999
print('list 切片修改后原 list =', lst)          # 原 list 不变

# 需要独立副本时用 .copy()
b = np.arange(6)
b_sub = b[2:5].copy()
b_sub[0] = 777
print('copy() 后修改，b =', b, '（不受影响）')
print('b_sub =', b_sub)

a   = [0 1 2 3 4 5]
sub = [2 3 4]
修改 sub[0]=999 之后 a = [  0   1 999   3   4   5]
np.shares_memory(a, sub) = True
list 切片修改后原 list = [0, 1, 2, 3, 4, 5]
copy() 后修改，b = [0 1 2 3 4 5] （不受影响）
b_sub = [777   3   4]


## 2.4 花式索引（fancy indexing）：用整数数组取值

除了切片，还可以用一个**整数数组 / 列表**作为索引，一次性取出多个位置的元素：

```python
idx = [3, 5, 7]
a[idx]   # 取出 a[3]、a[5]、a[7]
```

**花式索引与切片的关键区别：**

| 索引方式 | 写法 | 返回 | 共享内存？ |
|---|---|---|---|
| 切片 | `a[2:5]` | 视图 | 是 |
| 花式索引 | `a[[2, 3, 4]]` | **拷贝** | 否 |
| 布尔索引 | `a[mask]` | **拷贝** | 否 |

> ⚠️ **陷阱**：花式索引**返回拷贝**。想要「视图的效果」时用切片，想要「独立副本」时用花式索引——两者方向正好相反，务必分清。

**多维花式索引**：用两个长度相同的整数数组分别表示行、列，会**一一配对**取值——这正是取对角线 / 任意点的经典手法。

In [5]:
# 一维花式索引
a = np.arange(10)
idx = [3, 5, 7]
print('a =', a)
print('a[idx] =', a[idx])   # 取下标 3、5、7 的元素

# 花式索引返回【拷贝】，不是视图
f = a[[3, 5, 7]]
f[0] = -1
print('修改 f 后 a =', a)          # a 不受影响
print('shares_memory:', np.shares_memory(a, f))  # False

# 多维花式索引：取矩阵对角线 / 任意点
m = np.arange(16).reshape(4, 4)
print('m ：')
print(m)
rows = [0, 1, 2, 3]
cols = [0, 1, 2, 3]
print('m[rows, cols]（对角线） =', m[rows, cols])   # 行、列索引一一配对

print('m[[0, 2], [1, 3]] =', m[[0, 2], [1, 3]])     # 取 (0,1) 和 (2,3) 两个点

a = [0 1 2 3 4 5 6 7 8 9]
a[idx] = [3 5 7]
修改 f 后 a = [0 1 2 3 4 5 6 7 8 9]
shares_memory: False
m ：
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]
 [12 13 14 15]]
m[rows, cols]（对角线） = [ 0  5 10 15]
m[[0, 2], [1, 3]] = [ 1 11]


## 2.5 布尔索引：用「条件」筛选

**原理**：用一个与原数组**同形状的布尔数组**（mask）作为索引，返回所有 `True` 位置上的元素。

```mermaid
flowchart TD
    A[a 原始数组] --> B[mask 条件比较 a 大于 3]
    B --> C[同形状布尔数组 False False False True True]
    C --> D[a mask 筛选出满足条件的元素]
    D --> E{是否要赋值}
    E -->|要| F[a mask 赋值 原地修改原数组]
    E -->|不要| G[只读筛选结果]
```

**常见坑：多条件必须用 `&` `|` `~`，而且每个条件都要加括号！**

- ❌ `a[a > 0 and a < 5]` → 报错：`and` 要求布尔值，而这里是一整个数组
- ✅ `a[(a > 0) & (a < 5)]` → 正确

> ⚠️ **陷阱**：`and` / `or` 是 Python 逻辑运算符（短路、返回单一布尔），**不能**逐元素作用于数组；必须改用按位运算符 `&` / `|` / `~`。不写括号还会因为运算符优先级报错：`The truth value of an array ... is ambiguous`。

**布尔索引还能赋值**：`a[mask] = 值` 会修改所有满足条件的元素——例如把负数清零、把 NaN 替换。

In [6]:
# 基本筛选
a = np.array([1, 2, 3, 4, 5])
mask = a > 3
print('a =', a)
print('a > 3  ->', mask)              # 同形状布尔数组
print('mask 的 dtype =', mask.dtype)  # bool
print('a[mask] =', a[mask])           # 筛选出满足条件的元素

# 组合条件：必须用 & | ~，且每个条件都要加括号
b = np.array([1, -2, 3, -4, 5])
print('b =', b)
print('(b > 0) & (b < 5) ->', b[(b > 0) & (b < 5)])   # 且
print('(b < 0) | (b > 4) ->', b[(b < 0) | (b > 4)])   # 或
print('~(b == 3)         ->', b[~(b == 3)])           # 非

# 布尔索引赋值：把负数清零（数据清洗常用）
c = np.array([1, -2, 3, -4, 5])
c[c < 0] = 0
print('c[c<0]=0 后 c =', c)

a = [1 2 3 4 5]
a > 3  -> [False False False  True  True]
mask 的 dtype = bool
a[mask] = [4 5]
b = [ 1 -2  3 -4  5]
(b > 0) & (b < 5) -> [1 3]
(b < 0) | (b > 4) -> [-2 -4  5]
~(b == 3)         -> [ 1 -2 -4  5]
c[c<0]=0 后 c = [1 0 3 0 5]


## 2.6 np.where：条件的另一种打开方式

`np.where` 有三种常见用法：

| 用法 | 写法 | 作用 |
|---|---|---|
| 单参数 | `np.where(cond)` | 返回满足条件的**索引**（等价于 `np.nonzero(cond)`） |
| 三参数 | `np.where(cond, x, y)` | **向量化 if-else**：True 取 x，False 取 y |
| 嵌套 | 三参数里再套 `np.where` | 实现多分支选择 |

**为什么三参数版好用？** 对数组做 if-else，Python 的三元表达式 `a if cond else b` 只能处理**单个**布尔值，无法直接用于数组；`np.where` 一行搞定整个数组的逐元素判断。

> 💡 **提示**：`np.where` 只是「条件分支」的冰山一角，等学到布尔运算与逻辑函数（`np.logical_and` 等）会更完整。先掌握这里三种用法即可。

In [7]:
# 单参数：返回满足条件的【索引】
a = np.array([1, 2, 3, 4, 5])
print('np.where(a > 3)      =', np.where(a > 3))
print('np.nonzero(a > 3)    =', np.nonzero(a > 3))   # 等价写法

# 三参数：向量化 if-else
b = np.array([1, -2, 3, -4, 5])
print('b =', b)
print('np.where(b > 0, b, 0) =', np.where(b > 0, b, 0))   # 正数保留，否则置 0

# 嵌套 where 实现多分支：<0 -> -1，==0 -> 0，>0 -> 1（sign 效果）
c = np.array([-3, 0, 4, -1, 7])
sign = np.where(c > 0, 1, np.where(c < 0, -1, 0))
print('c       =', c)
print('sign(c) =', sign)
print('与 np.sign(c) 一致吗？', np.array_equal(sign, np.sign(c)))

np.where(a > 3)      = (array([3, 4]),)
np.nonzero(a > 3)    = (array([3, 4]),)
b = [ 1 -2  3 -4  5]
np.where(b > 0, b, 0) = [1 0 3 0 5]
c       = [-3  0  4 -1  7]
sign(c) = [-1  0  1 -1  1]
与 np.sign(c) 一致吗？ True


## 2.7 切片赋值与广播赋值

NumPy 允许把**标量**或**小数组**赋给切片 / 索引位置，值会被自动**广播**到整个目标区域：

- `a[0] = 5`：把第 0 个元素设为 5
- `a[2:4] = 8`：把 2、3 两个位置都设为 8（标量广播）
- `m[1] = 9`：把第 1 行整行设为 9
- `b[b < 0] = 0`：布尔赋值，把负数清零

> 💡 **提示**：布尔赋值是数据清洗的利器，比如把异常值、NaN、缺失值统一替换。

In [8]:
# 标量赋值：自动广播
arr1 = np.zeros(5, dtype=int)
print('arr1 =', arr1)
arr1[0] = 5       # 单个位置
arr1[2:4] = 8     # 一个标量广播到多个位置
print('arr1[0]=5, arr1[2:4]=8 后 arr1 =', arr1)

# 广播到整行
m = np.zeros((3, 4), dtype=int)
m[1] = 9          # 把第 1 行整行赋成 9
print('m ：')
print(m)

# 布尔赋值：把负值清零
b = np.array([1, -2, 3, -4, 5])
b[b < 0] = 0
print('布尔赋值后 b =', b)

# 布尔赋值：把 NaN 替换成 0
import numpy as np  # 已在上方导入，重复导入无副作用
d = np.array([1.0, np.nan, 3.0, np.nan])
d[np.isnan(d)] = 0.0
print('NaN 替换后 d =', d)

arr1 = [0 0 0 0 0]
arr1[0]=5, arr1[2:4]=8 后 arr1 = [5 0 8 8 0]
m ：
[[0 0 0 0]
 [9 9 9 9]
 [0 0 0 0]]
布尔赋值后 b = [1 0 3 0 5]
NaN 替换后 d = [1. 0. 3. 0.]


## 2.8 本章小结

### 一句话记忆表

| 主题 | 一句话记住 |
|---|---|
| 一维索引切片 | 与 list 一致：负索引从末尾数，`[start:stop:step]`，`[::-1]` 反转 |
| 多维索引 | 用 `m[i, j]`（一次定位），少用 `m[i][j]` |
| 切片 | **视图**：改切片 = 改原数组，用 `np.shares_memory` 验证，要独立用 `.copy()` |
| 花式索引 | 用整数数组 / 列表索引，返回**拷贝** |
| 布尔索引 | mask 是同形状布尔数组；多条件用 `&` `|` `~` 且加括号；可赋值 |
| np.where | 单参返回索引；三参 = 向量化 if-else，可嵌套多分支 |
| 赋值广播 | 标量赋给切片 / 整行 = 广播；布尔赋值改符合条件的元素 |

### 视图 vs 拷贝：判断口诀

> ⚠️ **口诀**：切片是视图（`arr[:2]`），花式索引是拷贝（`arr[[0, 1]]`），布尔索引是拷贝（`arr[mask]`），`astype` 和 `np.array` 是拷贝，`.copy()` 永远是拷贝。

知识地图：

```mermaid
flowchart TD
    A[取数方式] --> B{用数字还是条件}
    B -->|数字| C{切片还是花式}
    C -->|切片 范围| D[视图 共享内存]
    C -->|整数数组 列表| E[花式索引 拷贝]
    B -->|条件| F{几个条件}
    F -->|一个| G[布尔 mask 筛选]
    F -->|多个| H[逻辑运算符组合 需加括号]
    G --> I[a mask 取值]
    G --> J[a mask 赋值]
    A -.-> K[np.where 向量化 if else]
```

### 📝 动手练习（建议先自己写，再看提示）

1. **切片视图**：创建 `np.arange(10)`，用 `arr[2:8]` 得到一个视图并把它整体乘 2（`sub *= 2`），观察原数组发生了什么；再用 `.copy()` 复现一次，说明两者的区别。
2. **布尔筛选**：生成 10 个 0~1 之间的随机数（`np.random.default_rng().random(10)`），用布尔索引统计其中大于 0.5 的个数，并把这些数全部替换为 0。
3. **np.where 多分支**：对数组 `[-5, 0, 3, -1, 8]`，用嵌套 `np.where` 得到 `-1/0/1` 的符号数组，并验证与 `np.sign` 的结果一致。

> 💡 **练习提示**
>
> - 第 1 题：视图共享内存，`sub *= 2` 会让原数组对应位置一起翻倍；`.copy()` 则不会。
> - 第 2 题：`mask = r > 0.5`；`r[mask].size` 就是个数；`r[mask] = 0` 完成替换。
> - 第 3 题：`np.where(r > 0, 1, np.where(r < 0, -1, 0))`。

👉 **下一章**：`03_数组运算与广播.ipynb`（规划中，敬请期待）—— 将讲解算术运算、通用函数（ufunc）与广播（broadcasting）机制。